<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# Recuperación científica con TF-IDF y BM25

**Hito 1 · PLN · Máster en IA Aplicada · UC3M**

Antonio Bello, Martín Hernández, Iker Lekuona y Eduardo Jorge Sanjurjo

Comparamos **TF-IDF y BM25** sobre el mismo corpus y las mismas consultas: cuántos artículos recuperan, en qué posición aparecen y dónde fallan.

**Preparado, pendiente de ejecución con el índice completo.** Antes de evaluar, reconstruir los fragmentos siguiendo el README. Las salidas anteriores se han retirado para no confundirlas con resultados de esta versión.

</div>

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

### Ejecución

Instalar `requirements.txt` y `en_core_web_sm`. Colocar los textos completos en `data/txt/<tema>/` y ejecutar `python corpus_builder.py --check`; después, `python corpus_builder.py`. Ambos comandos funcionan sin red por defecto.

El notebook exige todos los fragmentos válidos y se detiene si encuentra el índice anterior de dos o tres por artículo. Mantener `REGENERATE_QUERIES = False` para comparar sobre el mismo test, sin Ollama ni clave de API. En Colab, configurar Drive y colocar `corpus_builder.py` junto al notebook.

</div>

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install pymupdf gensim spacy
    !python -m spacy download en_core_web_sm

In [ ]:
import os, sys, re, time, random, unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import requests
from scipy import sparse
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, HTML
from html import escape

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [8, 5]

# La tipografía se define en las propias celdas Markdown.
TOPIC_LABELS = {
    'autonomous_vehicles': 'Vehículos autónomos',
    'reinforcement_learning': 'Aprendizaje por refuerzo',
    'nlp': 'Procesamiento del lenguaje natural',
    'neural_networks': 'Redes neuronales',
    'deep_learning': 'Aprendizaje profundo',
}


def table_html(frame, caption=None, show_index=None):
    """Formato de informe con pandas.Styler; no modifica el DataFrame."""
    if show_index is None:
        show_index = not isinstance(frame.index, pd.RangeIndex)
    styles = [
        {'selector': '', 'props': [
            ('border-collapse', 'collapse'), ('width', '100%'),
            ('font-family', 'system-ui, sans-serif'), ('font-size', '14px'),
            ('line-height', '1.5'), ('color', 'var(--vscode-editor-foreground, var(--jp-ui-font-color1, #243247))'), ('background-color', 'var(--vscode-editor-background, var(--jp-layout-color0, #ffffff))')]},
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'), ('text-align', 'left'), ('font-weight', '600'),
            ('font-size', '15px'), ('padding', '14px 12px'),
            ('background-color', 'var(--vscode-editor-background, var(--jp-layout-color0, #ffffff))'), ('color', 'var(--vscode-editor-foreground, var(--jp-ui-font-color1, #243247))')]},
        {'selector': 'th', 'props': [
            ('background-color', 'var(--vscode-editorWidget-background, var(--jp-layout-color1, #eaf0f6))'), ('color', 'var(--vscode-editor-foreground, var(--jp-ui-font-color1, #243247))'),
            ('padding', '10px 12px'), ('text-align', 'left'),
            ('font-weight', '600'), ('border-bottom', '2px solid var(--vscode-panel-border, #b6c6d8)'),
            ('white-space', 'normal'), ('vertical-align', 'top')]},
        {'selector': 'thead th', 'props': [('position', 'sticky'), ('top', '0'), ('z-index', '1')]},
        {'selector': 'td', 'props': [
            ('padding', '9px 12px'), ('border-bottom', '1px solid var(--vscode-panel-border, #dce3eb)'),
            ('vertical-align', 'top'), ('text-align', 'left'),
            ('white-space', 'normal'), ('overflow-wrap', 'anywhere')]},
        {'selector': 'tbody tr:nth-child(even) td', 'props': [('background-color', 'var(--vscode-list-inactiveSelectionBackground, var(--jp-layout-color1, #f5f8fb))')]},
        {'selector': 'tbody tr:hover td', 'props': [('background-color', 'var(--vscode-list-hoverBackground, var(--jp-layout-color2, #eaf2fa))')]},
        {'selector': '.row_heading', 'props': [('font-weight', '500'), ('max-width', '180px')]},
    ]
    styled = (frame.style.format(precision=3, na_rep='—', escape='html')
              .format_index(precision=2, escape='html', axis=0)
              .format_index(precision=2, escape='html', axis=1)
              .set_table_styles(styles))
    numeric = frame.select_dtypes(include='number').columns.tolist()
    if numeric:
        styled = styled.set_properties(subset=numeric, **{
            'text-align': 'right', 'font-variant-numeric': 'tabular-nums',
            'white-space': 'nowrap'})
    text_columns = frame.select_dtypes(include=['object', 'string']).columns.tolist()
    if text_columns:
        styled = styled.set_properties(subset=text_columns, **{'max-width': '360px'})
    if not show_index:
        styled = styled.hide(axis='index')
    if caption:
        styled = styled.set_caption(escape(caption))
    return ('<div style="width:100%;max-width:1100px;max-height:520px;overflow:auto;'
            'background:var(--vscode-editor-background, var(--jp-layout-color0, #ffffff));'
            'color:var(--vscode-editor-foreground, var(--jp-ui-font-color1, #243247));'
            'margin:12px 0 24px;border:1px solid var(--vscode-panel-border, #dce3eb);border-radius:6px;">'
            + styled.to_html() + '</div>')


def show_table(frame, caption, show_index=None, collapsed=False):
    html = table_html(frame, caption, show_index)
    if collapsed:
        html = (f'<details style="margin:12px 0;"><summary style="cursor:pointer;">{escape(caption)}</summary>'
                + html + '</details>')
    display(HTML(html))


def show_reading(title, paragraphs):
    body = ''.join(f'<p style="margin:8px 0;">{escape(str(text))}</p>' for text in paragraphs)
    display(HTML('<section style="font-size:15px;line-height:1.7;max-width:88ch;margin:16px 0;">'
                 f'<strong>{escape(title)}</strong>{body}</section>'))


def show_example(title, fields, collapsed=False):
    """Texto de lectura con etiquetas y saltos de párrafo, sin representación de Python."""
    body = ''.join(
        '<div style="margin:12px 0;">'
        f'<div style="font-weight:600;margin-bottom:4px;">{escape(str(label))}</div>'
        f'<div style="white-space:pre-wrap;overflow-wrap:anywhere;">{escape(str(text))}</div></div>'
        for label, text in fields)
    if collapsed:
        html = (f'<details><summary style="cursor:pointer;font-weight:600;">{escape(title)}</summary>'
                + body + '</details>')
    else:
        html = f'<h4 style="margin:0 0 12px;">{escape(title)}</h4>' + body
    display(HTML('<section style="font-size:15px;line-height:1.7;max-width:88ch;margin:18px 0;'
                 'padding:16px;border:1px solid var(--vscode-panel-border, #ccd7e2);border-radius:6px;">' + html + '</section>'))

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 0 · Configuración

Los parámetros se fijan antes de evaluar. El conjunto de test no se utiliza para ajustarlos.

</div>

In [ ]:
SEED = 42

# Preprocesado
VALID_POS = {'NOUN', 'VERB', 'ADJ', 'PROPN'}
SPECIFIC_STW = {'paper', 'propose', 'present', 'show', 'use', 'method', 'approach', 'result', 'et', 'al',
                'fig', 'figure', 'table', 'section', 'arxiv', 'preprint', 'work', 'study', 'eq', 'equation',
                'appendix', 'github', 'http', 'https', 'www', 'com', 'org'}
PHRASES_MIN_COUNT, PHRASES_THRESHOLD = 5, 20
NO_BELOW, NO_ABOVE = 5, 0.5  # Frecuencia documental calculada sobre fragmentos.

# Consultas
N_CANDIDATES = 60
N_QUERIES_MIN = 30
MIN_QUERIES_PER_TOPIC = 5         # aviso si un tema se queda con menos consultas finales
MIN_REVIEWED = 10                 # revisión manual mínima exigida
LLM_BACKEND = 'uc3m'              # 'uc3m' (servidor del departamento) | 'ollama' (local)
LLM_MODEL = 'llama3.1:8b'
JUDGE_MODEL = 'llama3.1:8b'
UC3M_URL = 'https://yiyuan.tsc.uc3m.es/api/generate'
OLLAMA_URL = 'http://localhost:11434/api/generate'
VERIFY_SSL = False  # Activar cuando el servidor disponga de un certificado válido.
REGENERATE_QUERIES = False  # Regenerar invalida la revisión manual previa.
MIN_JUDGE_MATCH = 4  # Coherencia con el artículo, en escala de 1 a 5.
MIN_SHARED_TERMS = 2  # Mínimo de lemas compartidos con el título y el resumen.

# Recuperación y evaluación
BM25_K1, BM25_B = 1.5, 0.75
KS = (5, 10)
MAX_DEPTH = 100

In [ ]:
import subprocess, shutil
from urllib.parse import urlsplit

def _ollama_server_activo(timeout=2):
    endpoint = urlsplit(OLLAMA_URL)
    try:
        response = requests.get(f'{endpoint.scheme}://{endpoint.netloc}/api/tags', timeout=timeout)
        response.raise_for_status()
        return True
    except requests.RequestException:
        return False

def ensure_ollama_model(model, max_wait_server=60):
    """Prepara un modelo únicamente cuando se van a generar consultas nuevas."""
    if not _ollama_server_activo():
        if shutil.which('ollama') is None:
            raise RuntimeError(
                'Para generar consultas con este backend necesitas Ollama instalado y activo. '
                'Puedes usar LLM_BACKEND="uc3m" con tu clave, o conservar '
                'REGENERATE_QUERIES=False y las candidatas incluidas en data/.')
        endpoint = urlsplit(OLLAMA_URL)
        if endpoint.hostname not in ('localhost', '127.0.0.1', '::1'):
            raise RuntimeError(f'El servidor remoto de Ollama no responde: {OLLAMA_URL}')
        print('Arrancando el servidor de Ollama...')
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(max_wait_server):
            if _ollama_server_activo():
                break
            time.sleep(1)
        else:
            raise RuntimeError(f'El servidor de Ollama no respondió tras {max_wait_server} s.')

    endpoint = urlsplit(OLLAMA_URL)
    base_url = f'{endpoint.scheme}://{endpoint.netloc}'
    response = requests.get(base_url + '/api/tags', timeout=10)
    response.raise_for_status()
    installed = {item['name'] for item in response.json()['models']}
    if model not in installed and model + ':latest' not in installed:
        print(f'Descargando el modelo {model} (puede tardar varios minutos)...')
        response = requests.post(base_url + '/api/pull', json={'model': model, 'stream': False}, timeout=1800)
        response.raise_for_status()
        if response.json().get('error'):
            raise RuntimeError(response.json()['error'])
    else:
        print(f'Modelo {model} ya disponible')

def prepare_llm():
    if LLM_BACKEND == 'ollama':
        for model in dict.fromkeys([LLM_MODEL, JUDGE_MODEL]):
            ensure_ollama_model(model)
    elif LLM_BACKEND != 'uc3m':
        raise ValueError("LLM_BACKEND debe ser 'ollama' o 'uc3m'")

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
PROJECT_DIR = '.'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/NLP_Project/Entrega_H1'
sys.path.insert(0, PROJECT_DIR)

import corpus_builder
from corpus_builder import (TOPICS, PAPERS_PER_TOPIC, DATE_START, DATE_END,
                            CHUNK_WORDS, CHUNK_OVERLAP, URL_RE, EMAIL_RE, read_jsonl, write_jsonl)

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
OUT_DIR = PROJECT_DIR
CHUNKS_DIR = os.path.join(DATA_DIR, 'chunks')
MANIFEST_FILE = os.path.join(OUT_DIR, 'corpus.jsonl')
QUERIES_CAND_FILE = os.path.join(DATA_DIR, 'candidate_queries.jsonl')
QUERIES_FILE = os.path.join(OUT_DIR, 'queries.jsonl')
REVIEW_FILE = os.path.join(OUT_DIR, 'queries_review.csv')
os.makedirs(DATA_DIR, exist_ok=True)

def chunks_path(topic):
    return corpus_builder.chunks_path(CHUNKS_DIR, topic)

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 1 · Corpus

Se conserva la selección de 4 000 artículos de arXiv, 800 por tema, entre enero de 2025 y noviembre de 2026: vehículos autónomos, aprendizaje por refuerzo, PLN, redes neuronales y aprendizaje profundo.

El constructor utiliza los textos completos ya extraídos. Mantiene ventanas de 300 palabras, un solape del 15 %, un mínimo del 70 % de palabras alfabéticas y fragmentos residuales de al menos 100 palabras. Cada artículo debe conservar al menos 1 500 palabras y tres fragmentos válidos.

Se indexan **todos los fragmentos válidos**, sin límite global ni selección de dos o tres por artículo. Los identificadores mantienen la posición original de la ventana. La comprobación siguiente exige la misma selección de artículos y cobertura completa de sus fragmentos.

</div>

In [ ]:
required_files = [MANIFEST_FILE] + [chunks_path(topic) for topic in TOPICS]
missing = [path for path in required_files if not os.path.isfile(path)]
if missing:
    raise FileNotFoundError(
        f'Faltan archivos del corpus: {missing}. Prepara los TXT y ejecuta '
        '`python corpus_builder.py --check` y `python corpus_builder.py` antes del notebook.')

docs_df = pd.DataFrame(read_jsonl(MANIFEST_FILE))
chunks_df = pd.DataFrame([row for topic in TOPICS for row in read_jsonl(chunks_path(topic))])
if chunks_df.empty:
    raise ValueError('El índice está vacío: reconstruye los fragmentos desde los TXT.')
assert docs_df['doc_id'].is_unique, 'doc_id duplicados en el manifiesto'
assert chunks_df['chunk_id'].is_unique, 'chunk_id duplicados'
if set(chunks_df['doc_id']) != set(docs_df['doc_id']):
    raise ValueError('El índice debe conservar todos y solo los artículos de corpus.jsonl.')
per_doc = chunks_df.groupby('doc_id')
if not per_doc['n_chunks_doc'].nunique().eq(1).all():
    raise ValueError('Metadatos de fragmentos inconsistentes: reconstruye el índice completo.')
indexed_counts = per_doc.size()
expected_counts = per_doc['n_chunks_doc'].first()
if not indexed_counts.eq(expected_counts).all():
    raise ValueError(
        f'Índice parcial: {len(chunks_df):,} fragmentos indexados frente a '
        f'{int(expected_counts.sum()):,} válidos declarados. Ejecuta '
        '`python corpus_builder.py --check` y `python corpus_builder.py` antes de evaluar.')
docs_by_id = docs_df.set_index('doc_id').to_dict('index')

In [ ]:
summary = (docs_df.groupby('topic').size().rename('papers').to_frame()
           .join(chunks_df.groupby('topic').size().rename('fragmentos'))
           .join(chunks_df.groupby('topic')['n_words'].mean().round(0).rename('palabras/fragmento')))
summary.loc['TOTAL'] = [summary['papers'].sum(), summary['fragmentos'].sum(), chunks_df['n_words'].mean().round(0)]
show_table(summary.astype(int).rename(index=TOPIC_LABELS, columns={
    'papers': 'Artículos', 'fragmentos': 'Fragmentos',
    'palabras/fragmento': 'Palabras por fragmento'}).rename_axis('Tema'), 'Distribución del corpus por tema')

In [ ]:
import hashlib

# Inspección textual asistida: L=orden/segmentación, F=material editorial,
# N=contenido no discursivo, I=idioma distinto del modelo NLP.
# El hash vincula cada observación al texto revisado.
quality_cases = [('doc_2501.12269_c00', 'LF',
  'Resumen e introducción intercalados; secuencias como changespotentiallyin lighting.',
  '7866120e395474bb8d4dca4f21f29736c71c86ec62554f44861b8b5f5974963a'),
 ('doc_2504.07542_c00', 'LFN', 'Pie Figure 1 interrumpe el resumen; palabras partidas como Ger-Apr many.',
  '648979f328298d24abd42648b9948b48610f3cecd83bd40b9eade6a72800808b'),
 ('doc_2505.01446_c00', 'LF', 'Palabras fusionadas en vehiclesdata thatdohasnotbeenworkfed; restos de fecha.',
  '48a72e6e6ea8ec4fa84ea4037dd52668809aad2986ca1a1de58a9fb84509f66a'),
 ('doc_2508.04229_c06', 'LN', 'Etiquetas de agentes, diagrama y ecuaciones intercalados con el texto.',
  '56aadb0559b6ba7b699da75c0216bc10bd88f7585a4980b62979f0f7f7cad1c4'),
 ('doc_2510.13230_c24', 'N', 'Fórmulas de F1 y mAP aplanadas y número de página 15.',
  'f40938c914bb4af2d5d8e5c8173018caad7eb8f0cf9a7f82934a8ac5f946bd8d'),
 ('doc_2505.24415_c00', 'LF', 'Autores, highlights y fecha; secuencia Jul Addingual annotation.',
  '2e1b8c063f10d5ff58532547dbbff38832787e93817d9c803db755515f9cb943'),
 ('doc_2508.11181_c00', 'LF', 'Autores, fecha y título repetido; clasificación y porcentajes fusionados.',
  '5046915d11b2e5b05f65869511947834545c055dfd14a63672201bf92f10f005'),
 ('doc_2512.08264_c06', 'LN', 'Ecuaciones 11, 13, 14, 15 y secciones intercaladas.',
  '5d21b81af9100de72ee252f2993e00b4d10bd187c51f063f7daee7488d4468f5'),
 ('doc_2512.13018_c00', 'LF', 'Resumen y explicación de antecedentes intercalados; palabras cortadas.',
  '04244923dc94e9b940730cde2a04eb62f9ac5575c57580b3c6db1e1cd2131e26'),
 ('doc_2601.14605_c00', 'LF', 'Resumen y discusión intercalados; wenizationpropose y otras palabras fusionadas.',
  'c5a4a6ababa11533fd6eed435808362fd037cf6d05fa7bac251a6077a6782762'),
 ('doc_2505.13264_c00', 'LF', 'Resumen y motivación intercalados; autores, ORCID y fecha.',
  '6cedf2b4490db76c5da52da6f4ceafd1db409eff5055f44bfc20ce21a8fef062'),
 ('doc_2506.22771_c00', 'LF', 'Resumen y contribuciones intercalados; memoryThis paperfootprint fusionado.',
  '499eef938d1399bb0a6d170abcf3a38c427b1d76fd6bcc1a852ef3ee1e4c71ef'),
 ('doc_2507.00461_c06', 'LN', 'Ecuación, teorema y pie de figura intercalados; notación aplanada.',
  'acb5267322cb78f258429a89276a6a0763430e92cffeca2d26461649f1bbb54e'),
 ('doc_2605.27407_c05', 'N', 'Prosa mayormente legible; termina en una tabla de datasets aplanada.',
  '66b9b6e21e93fadfa9fac4a7e06451e21a418fe5124566ba9e2c270ac252f884'),
 ('doc_2611.12446_c19', 'N', 'Prosa mayormente legible; inicia con filas numéricas de resultados.',
  'ded33353739f66a27a1915f3019f7de026a933eb7ba278b3656eca6f0d1af3ff'),
 ('doc_2504.12177_c00', 'FI',
  'Resumen mayoritariamente en español, seguido de abstract inglés; el pipeline usa en_core_web_sm.',
  '2a81830320c2b0cedfa63fe652f7003c598ccf67182b9cfaaac71ca77ac49edb'),
 ('doc_2509.09701_c15', 'LN', 'Conclusión, resultados, pie de figura y regresión intercalados.',
  'e7f660695f64295105dd6dadd3c470bf7f39f9277efbb3c69bc8b647bdf3127f'),
 ('doc_2605.30727_c00', 'F', 'Prosa legible; conserva autores, afiliaciones, fecha y marca de categoría.',
  'dc8343fa82939cb87e2287467b9f049e35256bb6ea06b0cc1a8b84932d5e2578'),
 ('doc_2606.00272_c17', 'LN', 'Secciones 6.3 y 7.1 intercaladas; cabecera de conferencia y Table 4.',
  '581df5c9058758acf90f071f69bae0e84a25a50b28f8618e22d184ef03a4c689'),
 ('doc_2607.01115_c00', 'LF', 'Resumen e introducción intercalados; autores, afiliaciones y fecha.',
  '7fe97e139b4fe533a1e21d5b040926ca27ba085c9e61ef91d7f5b462eddf2edf'),
 ('doc_2501.04228_c07', 'LN', 'Diseño de restricciones y algoritmo intercalados, con numeración y ecuaciones.',
  '091994167c0b3b6d5d379dfafae80276355efcafb3919e4a0706a34e14371895'),
 ('doc_2601.18953_c00', 'FN',
  'Autores, afiliaciones, palabras partidas y tabla de contenidos mezclados con el resumen.',
  'df5f3bc6dd0ef7817b8167702e12920d4081be5a1f9d9eb34d22d7e1fef89d4e'),
 ('doc_2605.05755_c14', 'N', 'Prosa técnica legible; pie de figura y notación matemática aplanados.',
  'cc984826e7b35344c8d5e53717bffcb3683221cc806f946004fc5f886dd0441d'),
 ('doc_2607.22525_c00', 'F', 'Portada editorial, política de uso y referencia bibliográfica antes del resumen.',
  '7545eb59ab27c23c615178350175ae4e6ae28d0d59f036614a9d6c688a2a2700'),
 ('doc_2609.34344_c23', 'N',
  'El fragmento está compuesto por entradas bibliográficas, no por exposición del artículo.',
  'c0e87cd522ca1c0560347a1e624d279c4355e89b393a667ecf33ad307657319c')]
quality_review = pd.DataFrame(quality_cases, columns=['chunk_id', 'flags', 'observation', 'text_sha256'])
text_by_chunk = chunks_df.set_index('chunk_id')['text']
quality_review['unchanged'] = [
    chunk_id in text_by_chunk.index
    and hashlib.sha256(text_by_chunk[chunk_id].encode()).hexdigest() == digest
    for chunk_id, digest in zip(quality_review['chunk_id'], quality_review['text_sha256'])]
if quality_review['unchanged'].all():
    quality_flags = {
        'Orden de lectura o segmentación alterados': 'L',
        'Autores, afiliaciones o portada editorial': 'F',
        'Tablas, figuras, fórmulas aplanadas o bibliografía': 'N',
        'Idioma parcialmente distinto del modelo NLP': 'I',
    }
    show_table(pd.DataFrame([{'Hallazgo': label, 'Fragmentos afectados': int(quality_review['flags'].str.contains(flag).sum())}
                            for label, flag in quality_flags.items()]),
               'Inspección histórica: 25 fragmentos del índice anterior', show_index=False)
quality_sample = pd.concat([group.sample(n=min(5, len(group)), random_state=SEED)
                            for _, group in chunks_df.groupby('topic')], ignore_index=True)

def inspect_quality(chunk_id):
    row = chunks_df.set_index('chunk_id').loc[chunk_id]
    show_example(f'Inspección de {chunk_id}', [
        ('Artículo', docs_by_id[row['doc_id']]['title']), ('Texto completo del fragmento', row['text'])])

show_reading('Revisión del índice ampliado', [
    f"{int(quality_review['unchanged'].sum())} de los 25 textos históricos permanecen idénticos. "
    'Sus observaciones no representan todos los fragmentos nuevos. '
    'Revisa la nueva muestra quality_sample (hasta cinco fragmentos por tema) con inspect_quality(chunk_id). '
    'Si los textos han cambiado, las observaciones anteriores no se aplican automáticamente.'])

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 2 · Preprocesado

Fragmentos y consultas siguen el mismo proceso: limpieza de Unicode, enlaces y citas; lematización con spaCy; filtrado de palabras vacías; y conservación de sustantivos, verbos, adjetivos y nombres propios alfabéticos.

Gensim detecta expresiones como `reinforcement_learning`. El diccionario retiene términos presentes en al menos cinco fragmentos y no más del 50 % del índice. Las consultas no amplían ese vocabulario. El filtrado puede perder símbolos y términos técnicos útiles.

La tabla muestra un ejemplo de lematización; el extracto original puede desplegarse.

</div>

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

CITE_PAREN = re.compile(r'\([^()]{0,120}?\b(?:19|20)\d{2}[a-z]?\)')  # Ejemplo: (Smith et al., 2020)
CITE_BRACKET = re.compile(r'\[\s*\d+(?:\s*[,–-]\s*\d+)*\s*\]')  # Ejemplos: [12], [3, 5]

def wrangle_text(text):
    """Normaliza Unicode, elimina enlaces y citas y compacta espacios."""
    text = unicodedata.normalize('NFKC', text)
    for pattern in (URL_RE, EMAIL_RE, CITE_PAREN, CITE_BRACKET):
        text = pattern.sub(' ', text)
    return ' '.join(text.split())

def keep_token(tk):
    lemma = tk.lemma_.lower()
    return (tk.is_alpha and tk.pos_ in VALID_POS and not tk.is_stop
            and len(lemma) > 1 and lemma not in SPECIFIC_STW)

def prepare_data(texts, batch_size=64):
    """Devuelve una lista de lemas filtrados por cada texto."""
    wrangled = [wrangle_text(t) for t in texts]
    results = []
    for doc in tqdm(nlp.pipe(wrangled, batch_size=batch_size), total=len(wrangled), desc='spaCy', disable=len(wrangled) < 100):
        results.append([tk.lemma_.lower() for tk in doc if keep_token(tk)])
    return results

# Buscar un fragmento de cuerpo donde la limpieza tenga un efecto observable.
example_row = next((row for row in chunks_df.to_dict('records')
                    if row['chunk_idx'] > 0 and wrangle_text(row['text']) != row['text']),
                   chunks_df.iloc[0].to_dict())
example = example_row['text']
cleaned_example = wrangle_text(example)
common_prefix = len(os.path.commonprefix([example, cleaned_example]))
excerpt_start = max(0, common_prefix - 100)
example_excerpt = example[excerpt_start:excerpt_start + 600]
cleaned_excerpt = wrangle_text(example_excerpt)
show_example('Ejemplo de limpieza de un fragmento real', [
    ('Artículo de origen', docs_by_id[example_row['doc_id']]['title']),
    ('Extracto antes de la limpieza', example_excerpt),
    ('El mismo extracto después de la limpieza', cleaned_excerpt)], collapsed=True)
if example_excerpt == cleaned_excerpt:
    show_reading('Cambios en este extracto', [
        'No se han eliminado elementos en el tramo mostrado. La limpieza solo cambia el texto cuando encuentra alguno de los patrones definidos.'])
tokens = [(tk.text, tk.lemma_.lower(), tk.pos_, tk.is_stop, keep_token(tk))
          for tk in nlp(cleaned_excerpt)][:8]
token_view = pd.DataFrame(tokens, columns=['Palabra', 'Lema', 'Categoría', 'Palabra vacía', 'Se conserva'])
token_view['Categoría'] = token_view['Categoría'].replace({
    'NOUN': 'Sustantivo', 'VERB': 'Verbo', 'ADJ': 'Adjetivo', 'PROPN': 'Nombre propio',
    'ADP': 'Preposición', 'DET': 'Determinante', 'PUNCT': 'Puntuación', 'NUM': 'Número'})
for column in ['Palabra vacía', 'Se conserva']:
    token_view[column] = token_view[column].map({True: 'Sí', False: 'No'})
show_table(token_view, 'Lematización y filtrado: ocho unidades de un extracto real')

In [ ]:
LEMMAS_FILE = os.path.join(DATA_DIR, 'chunk_lemmas.jsonl')
signature = {'spacy': spacy.__version__, 'model': str(nlp.meta.get('version')), 'pos': sorted(VALID_POS),
             'stopwords': sorted(SPECIFIC_STW), 'chunks': len(chunks_df), 'first': chunks_df['chunk_id'].iloc[0],
             'index_sha256': corpus_builder.chunk_index_fingerprint(CHUNKS_DIR),
             'cleaning_patterns': [URL_RE.pattern, EMAIL_RE.pattern, CITE_PAREN.pattern, CITE_BRACKET.pattern]}

chunk_lemmas = None
if os.path.exists(LEMMAS_FILE):
    rows = read_jsonl(LEMMAS_FILE)
    if rows and rows[0] == signature and len(rows) == len(chunks_df) + 1:
        chunk_lemmas = rows[1:]
if chunk_lemmas is None:
    chunk_lemmas = prepare_data(chunks_df['text'].tolist())
    write_jsonl(LEMMAS_FILE, [signature] + chunk_lemmas)

In [ ]:
from gensim.models.phrases import Phrases
from gensim.corpora import Dictionary

phrase_model = Phrases(chunk_lemmas, min_count=PHRASES_MIN_COUNT, threshold=PHRASES_THRESHOLD)
corpus = [phrase_model[doc] for doc in chunk_lemmas]

D = Dictionary(corpus)
n_before = len(D)
D.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE, keep_n=None)
chunk_bow = [D.doc2bow(doc) for doc in corpus]

def preprocess(texts):
    """Pipeline completo para textos nuevos (consultas): limpieza -> spaCy -> n-gramas. El diccionario se aplica luego con D.doc2bow."""
    return [phrase_model[tokens] for tokens in prepare_data(texts)]

lengths = np.array([sum(c for _, c in bow) for bow in chunk_bow])
show_table(pd.DataFrame({
    'Magnitud': ['Vocabulario antes del filtro', 'Vocabulario final',
                 'Términos por fragmento (media)', 'Fragmentos sin términos'],
    'Valor': [f'{n_before:,}', f'{len(D):,}', f'{lengths.mean():.0f}', f'{(lengths == 0).sum()}']
}), 'Resumen del preprocesado')

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 3 · Consultas de evaluación

Cada consulta procede de un artículo conocido, que es **el único relevante anotado**. Otros artículos podrían responder bien y no contar como acierto.

El LLM recibe título, resumen y un fragmento. Se muestrean los cinco temas sin repetir artículos, combinando consultas breves, conversacionales y detalladas. Se generan más candidatas que las treinta consultas finales mínimas.

</div>

In [ ]:
AREA_NAMES = {'autonomous_vehicles': 'autonomous vehicles and self-driving', 'reinforcement_learning': 'reinforcement learning',
              'nlp': 'natural language processing', 'neural_networks': 'neural networks', 'deep_learning': 'deep learning'}

STYLES = {
    'simple': ("a very short phrase of AT MOST 5 words that names the main topic of the paper, the way someone would "
               "type a quick keyword search (for example: graph neural networks for molecules). Do not write a full "
               "sentence and do not exceed 5 words. Use plain, everyday words, not technical jargon.", 1, 5, 2),
    'conversational_statement': ("a natural sentence of 8 to 20 words that states what the researcher is looking into, "
                       "in the style of 'I am researching ...', 'I am looking for ...' or 'Papers about ...', stating "
                       "the single main idea of the paper as a researcher would say it out loud (for example: I am "
                       "researching methods to speed up protein folding simulations). IMPORTANT: use plain, everyday "
                       "language, as a non-expert would describe the general goal or problem. Do NOT use precise "
                       "technical terms, method names or jargon taken from the abstract; describe what the paper is "
                       "trying to do in general terms instead.", 6, 20, 2),
    'conversational_question': ("a natural QUESTION of 8 to 20 words that a researcher would ask out loud about the "
                       "paper's main idea (for example: Is there any work on speeding up protein folding simulations? "
                       "or Does anyone know a paper about ...?). IMPORTANT: use plain, everyday language, as a "
                       "non-expert would describe the general goal or problem. Do NOT use precise technical terms, "
                       "method names or jargon taken from the abstract; describe what the paper is trying to do in "
                       "general terms instead.", 6, 20, 2),
    'detailed': ("a query of 8 to 20 words, with the same natural, spoken tone as the conversational queries (NOT a "
                 "formal abstract sentence), but naming the specific problem and the particular method or contribution "
                 "of the paper in more depth. This style MAY use precise technical terms and method names.", 8, 20, 3),
}

def build_style_plan(n_total, seed, n_question=10, p_simple=0.15, p_detailed=0.05):
    """Reparto exacto de estilos: 15% simple, 5% detailed, el resto conversational
    (de las cuales como mucho `n_question` en formato de pregunta; el resto, declarativas)."""
    n_simple = round(p_simple * n_total)
    n_detailed = round(p_detailed * n_total)
    n_conv = n_total - n_simple - n_detailed
    n_q = min(n_question, n_conv)
    n_stmt = n_conv - n_q
    plan = (['simple'] * n_simple + ['detailed'] * n_detailed +
            ['conversational_question'] * n_q + ['conversational_statement'] * n_stmt)
    random.Random(seed).shuffle(plan)
    return plan

STYLE_PLAN = build_style_plan(N_CANDIDATES, SEED + 3)   # una etiqueta de estilo por candidata, en el orden en que se generan

GEN_PROMPT = """You are helping to build a test set for a search engine over scientific papers (arXiv).
You will receive the title and abstract of a paper and an excerpt from its full text.
Write ONE search query, in English, that a researcher who is looking for THIS paper could type.
Style of the query: {style_instruction}

Rules:
- The query must describe what the paper is about, so that it clearly belongs to the research area "{area}".
- Use the excerpt only to decide which topic of the paper to refer to; stay consistent with the paper as a whole.
- Use your own words: do NOT copy phrases from the title, the abstract or the excerpt; prefer paraphrases and synonyms.
- Do not include the title, author names, or numbers of sections, tables or figures.
- Output only the query, with no quotes and no explanation.

Research area: {area}
Title: {title}
Abstract: {abstract}
Excerpt:
{excerpt}
"""

JUDGE_PROMPT = """You are checking the quality of a test set for a search engine over scientific papers.
Research area: {area}
Paper title: {title}
Paper abstract: {abstract}

Search query: {query}

Rate the query with two integers from 1 to 5:
"match": how well the query describes THIS paper (1 = unrelated or about a different subject, 5 = clearly describes this paper's problem or contribution).
"specific": how specific the query is (1 = so generic that hundreds of papers in the area would match, 5 = it identifies a narrow set of papers).
Answer only with JSON, for example {{"match": 4, "specific": 3}}"""

_api_key = [None]

def api_key():
    """Obtiene la clave del entorno, de secretos de Colab o por teclado; la conserva en memoria."""
    if _api_key[0] is None:
        key = os.environ.get('UC3M_API_KEY')
        if not key and IN_COLAB:
            try:
                from google.colab import userdata
                key = userdata.get('UC3M_API_KEY')
            except Exception:
                key = None
        if not key:
            import getpass
            key = getpass.getpass('API key del servidor UC3M (está en el enunciado, sección 7): ')
        _api_key[0] = key
    return _api_key[0]

if not VERIFY_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def call_llm(prompt, model, temperature=0.2, retries=3):
    url = UC3M_URL if LLM_BACKEND == 'uc3m' else OLLAMA_URL
    headers = {'X-API-KEY': api_key()} if LLM_BACKEND == 'uc3m' else {}
    payload = {'model': model, 'prompt': prompt, 'stream': False, 'options': {'temperature': temperature, 'seed': SEED}}
    error = None
    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=180, verify=VERIFY_SSL)
            r.raise_for_status()
            return r.json()['response']
        except (requests.RequestException, KeyError, ValueError) as e:
            error = type(e).__name__
            time.sleep(2 ** attempt)
    raise RuntimeError(f'El LLM no responde ({error})')

def strip_thinking(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.S).strip()

def clean_query(text):
    lines = strip_thinking(text).splitlines()
    text = lines[0] if lines else ''
    text = re.sub(r'^\W*(search\s+)?query\s*:\s*', '', text, flags=re.I)
    return text.strip().strip('"\'`').strip()

def parse_judge(text):
    text = strip_thinking(text)
    m, s = re.search(r'"?match"?\s*:\s*([1-5])', text), re.search(r'"?specific"?\s*:\s*([1-5])', text)
    return (int(m.group(1)) if m else None, int(s.group(1)) if s else None)

def sample_source_chunks(n_total, seed):
    """Fragmentos al azar, repartidos a partes iguales entre temas y con un único fragmento por paper."""
    rng = random.Random(seed)
    topics = list(TOPICS)
    quota = {t: n_total // len(topics) + (i < n_total % len(topics)) for i, t in enumerate(topics)}
    picked = []
    for t in topics:
        pool = chunks_df[(chunks_df['topic'] == t) & (chunks_df['n_words'] >= 0.8 * CHUNK_WORDS)]
        doc_ids = sorted(pool['doc_id'].unique())
        rng.shuffle(doc_ids)
        for d in doc_ids[:quota[t]]:
            picked.append(pool[pool['doc_id'] == d].sample(1, random_state=rng.randrange(10 ** 6)).iloc[0])
    rng.shuffle(picked)
    return picked

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 3.1 · Controles automáticos

Se comprueban longitud, copia literal de cuatro palabras, lemas compartidos y valoraciones del LLM sobre coherencia y especificidad. `ok` supera todos los controles; `revisar` requiere inspección. Por defecto se reutilizan las candidatas guardadas.

</div>

In [ ]:
def generate_candidates():
    cands = []
    for i, row in enumerate(tqdm(sample_source_chunks(N_CANDIDATES, SEED + 2), desc='consultas'), 1):
        doc = docs_by_id[row['doc_id']]
        style = STYLE_PLAN[i - 1]
        fields = dict(area=AREA_NAMES[row['topic']], title=doc['title'], abstract=doc['abstract'])
        try:
            query = clean_query(call_llm(GEN_PROMPT.format(excerpt=row['text'], style_instruction=STYLES[style][0], **fields),
                                         LLM_MODEL, temperature=0.3))
            match, specific = parse_judge(call_llm(JUDGE_PROMPT.format(query=query, **fields), JUDGE_MODEL, temperature=0.0))
        except RuntimeError as e:
            print(f'  candidata {i} omitida: {e}')
            continue
        cands.append({'qid': f'q{i:02d}', 'query': query, 'relevant_ids': [row['doc_id']], 'source_chunk': row['chunk_id'],
                      'topic': row['topic'], 'style': style, 'judge_match': match, 'judge_specific': specific})
    return cands

queries_from_cache = os.path.exists(QUERIES_CAND_FILE) and not REGENERATE_QUERIES
if queries_from_cache:
    cands = read_jsonl(QUERIES_CAND_FILE)
else:
    prepare_llm()
    cands = generate_candidates()
    write_jsonl(QUERIES_CAND_FILE, cands)

In [ ]:
def words_of(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def ngram_copies(query, reference, n=4):
    """Nº de grupos de n palabras consecutivas de la consulta que aparecen literalmente en el texto de referencia."""
    q, r = words_of(query), words_of(reference)
    ref_grams = {tuple(r[i:i + n]) for i in range(len(r) - n + 1)}
    return sum(tuple(q[i:i + n]) in ref_grams for i in range(len(q) - n + 1))

chunk_text_by_id = dict(zip(chunks_df['chunk_id'], chunks_df['text']))
cand_by_id = {c['qid']: c for c in cands}

# Comparar lemas individuales, antes de formar n-gramas.
q_lemmas = prepare_data([c['query'] for c in cands])
a_lemmas = prepare_data([docs_by_id[c['relevant_ids'][0]]['title'] + '. ' + docs_by_id[c['relevant_ids'][0]]['abstract'] for c in cands])

for c, ql, al in zip(cands, q_lemmas, a_lemmas):
    doc = docs_by_id[c['relevant_ids'][0]]
    reference = doc['title'] + ' ' + doc['abstract']
    _, min_w, max_w, min_specific = STYLES[c['style']]
    c['n_words'] = len(c['query'].split())
    c['copy4'] = ngram_copies(c['query'], chunk_text_by_id[c['source_chunk']]) + ngram_copies(c['query'], reference)
    c['shared_terms'] = len(set(ql) & set(al))
    reasons = []
    if not min_w <= c['n_words'] <= max_w:
        reasons.append(f'longitud ({c["n_words"]} palabras; estilo {c["style"]}: {min_w}-{max_w})')
    if c['copy4'] > 0:
        reasons.append('copia literal de 4+ palabras')
    if c['shared_terms'] < MIN_SHARED_TERMS:
        
        reasons.append('casi sin términos del título/resumen')
    if c['judge_match'] is None or c['judge_match'] < MIN_JUDGE_MATCH:
        reasons.append(f'el LLM no ve que describa el paper (match={c["judge_match"]})')
    if c['judge_specific'] is None or c['judge_specific'] < min_specific:
        reasons.append(f'demasiado genérica (specific={c["judge_specific"]}, mínimo {min_specific})')
    c['auto'] = 'ok' if not reasons else 'revisar'
    c['reasons'] = '; '.join(reasons)

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 3.2 · Revisión manual

Se revisan al menos diez candidatas de los cinco temas, comprobando coherencia, especificidad y reformulación. Las decisiones de `REVIEW` prevalecen sobre los filtros; las demás consultas solo se aceptan si superan todos los controles.

`queries_review.csv` documenta las decisiones y `queries.jsonl` contiene el conjunto final que se vuelve a leer al evaluar. `inspect_query(qid)` permite consultar cada caso.

</div>

In [ ]:
def inspect_query(qid, expanded=True):
    c = cand_by_id[qid]
    doc = docs_by_id[c['relevant_ids'][0]]
    show_example(f"Revisión de {qid} · {TOPIC_LABELS[c['topic']]}", [
        ('Consulta', c['query']), ('Artículo de origen', doc['title']),
        ('Resumen del artículo', doc['abstract']),
        ('Fragmento que originó la consulta', chunk_text_by_id[c['source_chunk']]),
        ('Motivo de revisión', c['reasons'] or 'Supera los controles automáticos.'),
        ('Valoración del LLM', f"Coherencia: {c['judge_match']}/5. Especificidad: {c['judge_specific']}/5.")
    ], collapsed=not expanded)

In [ ]:
REVIEW = {
    'q09': ('discard', 'el paper es un survey de 4 dominios (power, AV, HPC, redes); la consulta solo menciona coches autónomos y da una idea equivocada del alcance'),
    'q13': ('discard', 'tema distinto: el paper trata comunicación/redes cuánticas, no amenazas de seguridad de ordenadores cuánticos (coincide con match=1)'),
    'q15': ('ok',      'buena paráfrasis; el 4-grama compartido es terminología genérica del campo, no identifica el paper de forma trivial'),
    'q18': ('ok',      'buena paráfrasis; el 4-grama compartido es una expresión académica genérica ("evaluate and compare different")'),
    'q23': ('ok',      'excelente paráfrasis (match=5, specific=5); se pasa solo 2 palabras del límite de longitud, aceptable'),
    'q24': ('ok',      'describe bien el tema; el aviso es solo de longitud (22 vs 20 palabras)'),
    'q26': ('discard', 'repite casi literalmente la frase del título ("building density and height")'),
    'q28': ('discard', 'repite literalmente una frase del título ("off-road autonomous driving")'),
    'q29': ('ok',      'buena paráfrasis; el 4-grama compartido es terminología habitual del área ("web information seeking")'),
    'q32': ('ok',      'buena descripción; "real-world control problems" es término genérico de RL, no exclusivo del paper'),
    'q51': ('discard', 'repite literalmente la frase del título ("vehicle detection and tracking")'),
    'q53': ('discard', 'demasiado genérica: "self-driving car safety improvements" valdría para cualquier paper de AV sobre seguridad'),
}

In [ ]:
unknown = [q for q in REVIEW if q not in cand_by_id]
bad_values = [q for q, (decision, _) in REVIEW.items() if decision not in ('ok', 'discard')]
assert not unknown, f'qid inexistentes en REVIEW: {unknown}'
assert not bad_values, f"la decisión debe ser 'ok' o 'discard': {bad_values}"

def decide(c):
    """Decisión humana si la hay; si no, sólo se conserva lo que supera todos los controles automáticos."""
    if c['qid'] in REVIEW:
        return REVIEW[c['qid']][0]
    return 'ok' if c['auto'] == 'ok' else 'discard'

for c in cands:
    c['final'] = decide(c)
kept = [c for c in cands if c['final'] == 'ok']

write_jsonl(QUERIES_FILE, [{'qid': c['qid'], 'query': c['query'], 'relevant_ids': c['relevant_ids'], 'source_chunk': c['source_chunk']}
                           for c in kept])
pd.DataFrame([{'qid': c['qid'], 'topic': c['topic'], 'query': c['query'], 'auto': c['auto'], 'auto_reasons': c['reasons'],
               'judge_match': c['judge_match'], 'judge_specific': c['judge_specific'],
               'manual_decision': REVIEW.get(c['qid'], ('', ''))[0], 'manual_note': REVIEW.get(c['qid'], ('', ''))[1],
               'final': c['final']} for c in cands]).astype({'judge_match': 'Int64', 'judge_specific': 'Int64'}).to_csv(REVIEW_FILE, index=False)

n_reviewed = len(REVIEW)
per_topic = Counter(c['topic'] for c in kept)
query_summary = pd.DataFrame([{
    'Tema': TOPIC_LABELS[t],
    'Candidatas': sum(c['topic'] == t for c in cands),
    'Revisadas': sum(c['topic'] == t and c['qid'] in REVIEW for c in cands),
    'Aceptadas': per_topic.get(t, 0)} for t in TOPICS])
show_table(query_summary, 'Selección de consultas por tema')
if n_reviewed < MIN_REVIEWED:
    print(f'AVISO: FALTA LA REVISIÓN MANUAL: {n_reviewed} de {MIN_REVIEWED} consultas revisadas. Completad REVIEW y volved a ejecutar desde la celda de REVIEW hasta el final.')
if len(kept) < N_QUERIES_MIN:
    print(f'AVISO: sólo {len(kept)} consultas válidas (mínimo {N_QUERIES_MIN}): subid N_CANDIDATES y regenerad (REGENERATE_QUERIES=True).')
if any(per_topic.get(t, 0) < MIN_QUERIES_PER_TOPIC for t in TOPICS):
    print(f'AVISO: algún tema tiene menos de {MIN_QUERIES_PER_TOPIC} consultas: el conjunto de test no cubre bien los 5 temas.')

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 4 · Recuperación

Ambos métodos puntúan los mismos fragmentos mediante términos compartidos. La diferencia está en la ponderación, no en el preprocesado.

</div>

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 4.1 · TF-IDF y coseno

Se utiliza Gensim con frecuencia bruta e IDF logarítmico:

$$w_{t,c}=f(t,c)\log_2\left(\frac{N}{\operatorname{df}(t)}\right).$$

$f(t,c)$ es la frecuencia del término; $N$, el número de fragmentos; y $\operatorname{df}(t)$, cuántos lo contienen. Consulta y fragmentos se normalizan a norma L2: su producto escalar equivale al coseno.

</div>

In [ ]:
from gensim.models import TfidfModel
from gensim.matutils import corpus2csc

tfidf = TfidfModel(chunk_bow)
n_chunks, n_terms = len(chunk_bow), len(D)

# Filas: fragmentos. Columnas: términos.
X_counts = corpus2csc(chunk_bow, num_terms=n_terms, num_docs=n_chunks).T.tocsr()
X_tfidf = corpus2csc(tfidf[chunk_bow], num_terms=n_terms, num_docs=n_chunks).T.tocsr()

def tfidf_scores(tokens):
    """Similitud coseno entre una consulta (lista de términos) y todos los fragmentos."""
    vec = np.zeros(n_terms)
    for term_id, weight in tfidf[D.doc2bow(tokens)]:
        vec[term_id] = weight
    return X_tfidf @ vec

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 4.2 · BM25

Implementamos Okapi BM25 con IDF de Lucene:

$$s(q,c)=\sum_{t\in\operatorname{unique}(q)}\operatorname{IDF}(t)\frac{f(t,c)(k_1+1)}{f(t,c)+k_1\left(1-b+b\frac{|c|}{\operatorname{avgdl}}\right)}.$$

$$\operatorname{IDF}(t)=\ln\left(1+\frac{N-\operatorname{df}(t)+0.5}{\operatorname{df}(t)+0.5}\right).$$

$k_1=1.5$ regula la saturación y $b=0.75$, la normalización por longitud. $|c|$ cuenta términos retenidos y $\operatorname{avgdl}$ es su media. Todas las estadísticas se calculan sobre fragmentos.

</div>

In [ ]:
class BM25:
    """Okapi BM25 (IDF de Lucene) sobre una matriz dispersa de conteos fragmentos x términos."""
    def __init__(self, counts, k1=1.5, b=0.75):
        counts = sparse.csr_matrix(counts)
        self.k1, self.b = k1, b
        self.n_docs = counts.shape[0]
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(float)
        self.avgdl = self.doc_len.mean()
        df = np.asarray((counts > 0).sum(axis=0)).ravel()
        self.idf = np.log1p((self.n_docs - df + 0.5) / (df + 0.5))
        coo = counts.tocoo()
        tf = coo.data.astype(float)
        norm = k1 * (1 - b + b * self.doc_len[coo.row] / self.avgdl)
        weights = self.idf[coo.col] * tf * (k1 + 1) / (tf + norm)
        self.W = sparse.csc_matrix((weights, (coo.row, coo.col)), shape=counts.shape)

    def scores(self, term_ids):
        """Puntuación de todos los fragmentos para una consulta (ids de sus términos distintos)."""
        term_ids = sorted(set(term_ids))
        if not term_ids:
            return np.zeros(self.n_docs)
        return np.asarray(self.W[:, term_ids].sum(axis=1)).ravel()

bm25 = BM25(X_counts, k1=BM25_K1, b=BM25_B)
bm25_label = f'BM25 (k1={BM25_K1}, b={BM25_B})'

def query_term_ids(tokens):
    return [term_id for term_id, _ in D.doc2bow(tokens)]

def bm25_scores(tokens):
    return bm25.scores(query_term_ids(tokens))

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 5 · Evaluación

Cada artículo recibe la puntuación de su mejor fragmento: $s(q,d)=\max_{c\in C(d)}s(q,c)$. Se ordenan todos los artículos con puntuación positiva; los empates siguen un orden estable.

**Recall@5 y Recall@10** miden la proporción de consultas que recupera el artículo dentro de esos cortes. **MRR** promedia el inverso de su posición en el ranking completo; si no aparece, aporta cero. Un valor mayor es mejor. Los cortes MRR@10 y MRR@100 quedan calculados en `results_df` como diagnóstico complementario.

</div>

In [ ]:
queries = read_jsonl(QUERIES_FILE)
assert queries, 'queries.jsonl está vacío: revisad la sección 3'
query_tokens = preprocess([q['query'] for q in queries])
relevants = [set(q['relevant_ids']) for q in queries]
q_topic = [docs_by_id[q['relevant_ids'][0]]['topic'] for q in queries]


doc_ids = docs_df['doc_id'].tolist()
doc_index = {d: i for i, d in enumerate(doc_ids)}
chunk_doc_idx = chunks_df['doc_id'].map(doc_index).to_numpy()

def doc_scores_from_chunks(chunk_scores):
    """Asigna a cada artículo la puntuación máxima de sus fragmentos."""
    scores = np.full(len(doc_ids), -np.inf)
    np.maximum.at(scores, chunk_doc_idx, chunk_scores)
    return scores

def top_docs(chunk_scores, k=None):
    """Ordena artículos únicos con puntuación positiva; k=None conserva el ranking completo."""
    scores = doc_scores_from_chunks(chunk_scores)
    order = np.argsort(-scores, kind='stable')
    if k is not None:
        order = order[:k]
    return [doc_ids[i] for i in order if scores[i] > 0]


def recalls_at_k(rankings, relevants, k):
    """Recall@K de cada consulta."""
    return np.array([len(set(r[:k]) & rel) / len(rel) for r, rel in zip(rankings, relevants)])

def recall_at_k(rankings, relevants, k):
    return float(recalls_at_k(rankings, relevants, k).mean())

def reciprocal_ranks(rankings, relevants, k):
    out = []
    for r, rel in zip(rankings, relevants):
        out.append(next((1 / pos for pos, d in enumerate(r if k is None else r[:k], 1) if d in rel), 0.0))
    return np.array(out)

def mrr(rankings, relevants, k=None):
    return float(reciprocal_ranks(rankings, relevants, k).mean())


# Validar las métricas con rankings de resultado conocido.
_r = [['a', 'b', 'c'], ['x', 'y', 'q'], ['p', 'r', 's']]
_rel = [{'b'}, {'q'}, {'zzz'}]
assert recall_at_k(_r, _rel, 1) == 0.0 and abs(recall_at_k(_r, _rel, 2) - 1 / 3) < 1e-12 and abs(recall_at_k(_r, _rel, 3) - 2 / 3) < 1e-12
assert abs(mrr(_r, _rel, 3) - (1 / 2 + 1 / 3 + 0) / 3) < 1e-12
assert abs(mrr(_r, _rel) - (1 / 2 + 1 / 3) / 3) < 1e-12

In [ ]:
METHODS = {'TF-IDF + coseno': tfidf_scores, bm25_label: bm25_scores}
S = {name: np.vstack([fn(t) for t in query_tokens]) for name, fn in METHODS.items()}  # Filas: consultas. Columnas: fragmentos.

def evaluate(scores):
    doc_rank = [top_docs(s) for s in scores]
    res = {f'Recall@{k}': recall_at_k(doc_rank, relevants, k) for k in KS}
    res['MRR'] = mrr(doc_rank, relevants)
    res['MRR@10'] = mrr(doc_rank, relevants, 10)
    res['MRR@100'] = mrr(doc_rank, relevants, MAX_DEPTH)
    return res, doc_rank

evaluated = {name: evaluate(S[name]) for name in S}
doc_rankings = {name: ev[1] for name, ev in evaluated.items()}
results_df = pd.DataFrame({name: ev[0] for name, ev in evaluated.items()}).T

show_table(results_df[['Recall@5', 'Recall@10', 'MRR']].T, 'Recuperación de artículos')

method_names = list(results_df.index)
first_method, second_method = method_names
recovered10 = {name: sum(bool(set(r[:10]) & rel) for r, rel in zip(doc_rankings[name], relevants))
               for name in method_names}
delta = results_df.loc[second_method, 'Recall@10'] - results_df.loc[first_method, 'Recall@10']
show_reading('Resultado principal', [
    f"TF-IDF recupera {recovered10[first_method]} de {len(queries)} artículos en top 10 "
    f"({results_df.loc[first_method, 'Recall@10']:.1%}); BM25 recupera {recovered10[second_method]} "
    f"({results_df.loc[second_method, 'Recall@10']:.1%}). La diferencia BM25 − TF-IDF es "
    f"{delta * 100:+.1f} puntos porcentuales. Los intervalos siguientes describen su incertidumbre."])

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

### Referencia anterior: índice de 10 000 fragmentos

| Método | Recall@5 | Recall@10 | MRR completo |
|---|---:|---:|---:|
| TF-IDF + coseno | 0,407 | 0,574 | 0,275 |
| BM25 | 0,630 | 0,741 | 0,533 |

Resultados históricos sobre 54 consultas, con los mismos parámetros. No corresponden al índice ampliado. La comparación requiere conservar consultas y preprocesado.

</div>

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 5.1 · Incertidumbre y temas

Se calculan intervalos percentiles del 95 % con 5 000 remuestreos bootstrap de la diferencia por consulta, **BM25 − TF-IDF**. Un intervalo positivo favorece a BM25; si incluye cero, la diferencia no es concluyente.

El desglose por tema debe leerse junto al número de consultas. Los intervalos describen este conjunto de prueba, no toda posible variación del corpus.

</div>

In [ ]:
names = list(S)
hit10 = {n: recalls_at_k(doc_rankings[n], relevants, 10) for n in names}
rr = {n: reciprocal_ranks(doc_rankings[n], relevants, None) for n in names}

def bootstrap(values, n_boot=5000, seed=SEED):
    """Media e intervalo de confianza al 95 % por bootstrap sobre las consultas."""
    values = np.asarray(values, float)
    idx = np.random.default_rng(seed).integers(0, len(values), (n_boot, len(values)))
    lo, hi = np.percentile(values[idx].mean(axis=1), [2.5, 97.5])
    return values.mean(), lo, hi

a, b = names
comparisons = []
for label, (va, vb) in {'Recall@10': (hit10[a], hit10[b]), 'MRR': (rr[a], rr[b])}.items():
    m, lo, hi = bootstrap(vb - va)
    interpretation = ('Favorece a BM25' if lo > 0 else
                      'Favorece a TF-IDF' if hi < 0 else 'Diferencia no concluyente')
    comparisons.append({'Métrica': label, 'Diferencia': m, 'Límite inferior': lo,
                        'Límite superior': hi, 'Lectura': interpretation})
show_table(pd.DataFrame(comparisons), 'Diferencia BM25 − TF-IDF · intervalo del 95 %')

short = {a: 'TF-IDF', b: 'BM25'}
rows = []
for t in TOPICS:
    idx = [i for i, qt in enumerate(q_topic) if qt == t]
    if idx:
        row = {'tema': t, 'consultas': len(idx)}
        for n in names:
            row[f'Recall@10 {short[n]}'] = hit10[n][idx].mean()
            row[f'MRR {short[n]}'] = rr[n][idx].mean()
        rows.append(row)
topic_results = pd.DataFrame(rows).set_index('tema').rename(index=TOPIC_LABELS)
show_table(topic_results.round(3).rename(columns={'consultas': 'Consultas'}), 'Resultados por tema')

In [ ]:
from matplotlib.ticker import PercentFormatter

ks = np.arange(1, 51)
recall_curves = {name: np.array([recall_at_k(doc_rankings[name], relevants, int(k)) for k in ks])
                 for name in names}
with plt.rc_context({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.facecolor': 'white', 'figure.facecolor': 'white',
                     'text.color': '#172b4d', 'axes.labelcolor': '#334155',
                     'xtick.color': '#475569', 'ytick.color': '#475569'}):
    fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
    colors = {a: '#2563eb', b: '#d95f02'}
    for name, marker in [(a, 'o'), (b, 'D')]:
        ax.step(ks, recall_curves[name], where='post', color=colors[name],
                linewidth=2.6, label=short[name], zorder=3)
        for cutoff in (5, 10):
            ax.scatter(cutoff, recall_curves[name][cutoff - 1], s=44,
                       marker=marker, color=colors[name], edgecolor='white', zorder=4)
    for cutoff in (5, 10):
        ax.axvline(cutoff, color='#94a3b8', linewidth=1, linestyle=(0, (3, 4)), zorder=1)
    for name, offset in [(a, (10, -24)), (b, (10, 10))]:
        value = recall_curves[name][9]
        ax.annotate(f'{short[name]}: {value:.1%} en K=10', (10, value),
                    xytext=offset, textcoords='offset points', color=colors[name],
                    fontsize=10, fontweight='semibold',
                    bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': .9, 'pad': 2})
    ax.set(title='Recuperación de artículos al ampliar el ranking',
           xlabel='Número de resultados considerados (K)', ylabel='Consultas con artículo recuperado',
           xlim=(1, 50), ylim=(0, 1))
    ax.set_xticks([1, 5, 10, 20, 30, 40, 50])
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', color='#e2e8f0', linewidth=.8)
    ax.grid(axis='x', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('#cbd5e1')
    ax.legend(loc='lower right', frameon=False)
    plt.show()

same_budget_gain = float(recall_curves[b][9] - recall_curves[a][9])
reading = (f'Con diez resultados, BM25 recupera {recall_curves[b][9]:.1%} de los artículos y TF-IDF '
           f'{recall_curves[a][9]:.1%}: {same_budget_gain * 100:+.1f} puntos porcentuales. '
           f'Al ampliar hasta cincuenta resultados, alcanzan {recall_curves[b][-1]:.1%} y '
           f'{recall_curves[a][-1]:.1%}, respectivamente: la ventaja pasa a '
           f'{(recall_curves[b][-1] - recall_curves[a][-1]) * 100:.1f} puntos. '
           'Los escalones reflejan consultas adicionales resueltas; los tramos planos no añaden aciertos. '
           'La curva mide cobertura del artículo de origen, no la relevancia de todos los resultados.')
show_reading('Interpretación de la gráfica', [reading])

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

# 6 · Análisis de fallos

Se identifican las consultas donde ambos métodos fallan en top 10. La tabla incluye sus posiciones y distingue las observaciones históricas todavía aplicables de los casos pendientes de revisión. Para estos últimos hay que contrastar consulta, resumen de origen y nuevos competidores.

Las categorías son hipótesis orientativas, no causas demostradas ni juicios exhaustivos de relevancia. El cambio de índice puede modificar tanto los fallos como sus explicaciones.

</div>

In [ ]:
hits_tfidf = hit10[a].astype(bool)
hits_bm25 = hit10[b].astype(bool)
show_reading('Desacuerdo entre métodos', [
    f'En top 10, solo BM25 acierta en {int((~hits_tfidf & hits_bm25).sum())} consultas y solo TF-IDF '
    f'en {int((hits_tfidf & ~hits_bm25).sum())}. Ambos fallan en {int((~hits_tfidf & ~hits_bm25).sum())}.'])

In [ ]:
# La firma identifica la consulta, el relevante y el primer competidor de cada método.
failure_cases = [('q03', 'Consulta amplia',
  'La consulta omite las monedas; el artículo estudia coin grading y el primer competidor de BM25 también compara '
  'características y deep learning.',
  '2be0ff1ad6aaa8816be5f229bd383a2669e3c19bc979ab20b43290334ee2480e'),
 ('q11', 'Consulta amplia',
  'Se pide adaptación al input sin mencionar neuromodulación ni hardware mixed-signal, que delimitan la '
  'contribución del artículo.',
  'fa02edb16e054f0b684f2c54aa49d594d6b86d9f0945b213cce89cde01bc3f64'),
 ('q15', 'Competidor plausible',
  'El primer resultado de ambos métodos estudia FPGA neuromórfica y reducción de potencia; podría ser útil aunque '
  'no sea el artículo Catwalk.',
  'd96bcf317d274d22153e19ce5a09faeca7a34edefdb913cdb53a9e072d2c4d61'),
 ('q18', 'Desajuste léxico',
  'La consulta omite external human-machine interfaces; los primeros resultados tratan planificación o '
  'conducción, no evaluación de eHMI.',
  'f4e51cb3c5644f734a839a141018023469ebbce0d2e02a2c5c981646ba8b7598'),
 ('q24', 'Desajuste léxico',
  'La consulta dice adjust settings sin nombrar hiperparámetros; el artículo estudia cotas de complejidad de su '
  'ajuste.',
  '93d4410196d3323b204b109f925a7d49d8d464b9dd2bd866e57cd934e12ec23e'),
 ('q34', 'Desajuste léxico',
  'La consulta habla de vehículos hidden from view; el artículo articula la tarea como occlusion-aware '
  'contingency planning.',
  '1ecab288e10b655a72f73aa6a878e99bb5f0d1683e1463665003816c94a9aac6'),
 ('q37', 'Consulta amplia',
  'La consulta omite control por eventos y ramp metering; los competidores tratan tráfico o conducción sin '
  'identificar ese mecanismo.',
  '41c249343925dc4abb1263679da6a8dfc49669ef2624186d0fa47df7fe32d01f'),
 ('q46', 'Desajuste léxico',
  'Switch to different computers no nombra migración de agent twins entre RSU; los primeros resultados describen '
  'conducción o flujo de tráfico.',
  '2c59fa611b1c8689ce7540a7636f1c7d420063ada975c787c52745cbaaf29f45'),
 ('q47', 'Desajuste léxico',
  'La consulta menciona prior knowledge; el artículo concreta biokinetic ODE priors y los competidores usan '
  'priors en otros dominios.',
  'ec416e0caf7c61a99b123bcf12711bfc66a4068ac1c178365f9d2b03d34e2753'),
 ('q52', 'Consulta amplia',
  'Cuatro palabras omiten confrontación multi-UGV y comandante VLM; el primer resultado TF-IDF sobre equipos de '
  'robots es temáticamente próximo.',
  'ae571588ff39c06802bfca0e544ae59d4828913e02fde131b076bc099e5024ab'),
 ('q55', 'Desajuste léxico',
  'La consulta omite symbolic regression; ambos métodos priorizan un artículo sobre learning rules y estructura '
  'jerárquica de datos.',
  '75137ca6d6db84951d3b68e37112f583d8eef77c1953cc82d4f90076f0e57019'),
 ('q59', 'Competidor plausible',
  'El primer resultado BM25, Bruno, también trata entrenamiento hardware-aware de neuronas spiking; su utilidad '
  'exigiría un juicio adicional.',
  '8ab6e55ea0cadd95d88125e116ffe10f31901d859514e1996a69c9f2a7085a04'),
 ('q60', 'Consulta amplia',
  'La consulta no identifica routing ni computación iterativa; el artículo H-Model es un prototipo y los '
  'competidores abordan adaptación de redes.',
  'de6f72eaeab7231ed6f1ad6edee6aae7432ad918378eab9208d365ec513acfb7')]
historical_failures = pd.DataFrame(failure_cases, columns=['qid', 'category', 'evidence_and_hypothesis', 'review_sha256']).set_index('qid')
failed_indices = np.flatnonzero(~hits_tfidf & ~hits_bm25)
failure_rows = []
for i in failed_indices:
    query = queries[i]
    relevant = query['relevant_ids'][0]
    top_a = doc_rankings[a][i][0] if doc_rankings[a][i] else ''
    top_b = doc_rankings[b][i][0] if doc_rankings[b][i] else ''
    signature = hashlib.sha256('\n'.join([query['query'], relevant, top_a, top_b]).encode()).hexdigest()
    category, evidence = 'Revisión pendiente', 'Contrastar la consulta, el resumen de origen y los nuevos competidores.'
    if query['qid'] in historical_failures.index:
        previous = historical_failures.loc[query['qid']]
        if signature == previous['review_sha256']:
            category, evidence = previous['category'], previous['evidence_and_hypothesis']
    row = {'ID': query['qid'], 'Hipótesis principal': category, 'Evidencia e interpretación': evidence}
    for name in names:
        ranking = doc_rankings[name][i]
        row[short[name]] = ranking.index(relevant) + 1 if relevant in ranking else 'No recuperado'
    failure_rows.append(row)
failure_review = pd.DataFrame(failure_rows, columns=['ID', 'TF-IDF', 'BM25', 'Hipótesis principal', 'Evidencia e interpretación'])
if failure_review.empty:
    show_reading('Análisis de fallos', ['Ninguna consulta falla en top 10 con ambos métodos.'])
else:
    show_table(failure_review, 'Fallos compartidos del índice actual', show_index=False)
    pending = int(failure_review['Hipótesis principal'].eq('Revisión pendiente').sum())
    show_reading('Estado de la revisión', [
        f'{len(failure_review)} fallos compartidos; {pending} requieren una nueva revisión. '
        'Las notas históricas se reutilizan solo si coinciden consulta, relevante y primeros competidores. '
        'No se modifican las consultas para mejorar los resultados del test.'])

<div style="max-width:88ch;font-size:16px;line-height:1.75;padding:4px 0 12px;">

## 6.1 · Limitaciones

Los resultados describen este índice y este conjunto de consultas. Exigir términos compartidos puede favorecer a los métodos léxicos; generador y juez usan el mismo modelo. Los competidores plausibles necesitarían juicios adicionales de relevancia.

Conservar todos los fragmentos amplía la cobertura, pero no corrige el orden de lectura, los restos editoriales ni los problemas de idioma del texto extraído. La auditoría histórica debe complementarse con una inspección del índice ampliado. La calidad de recuperación solo puede valorarse después de recalcular las métricas.

</div>